# 自注意力机制（Self-Attention）手撕实现

## 1. 核心
对输入序列 $X\in\mathbb{R}^{n\times d}$，用三个投影得到 $Q=XW_Q,\ K=XW_K,\ V=XW_V$：
$$\text{Attention}(Q,K,V)=\text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$
- **为什么除 $\sqrt{d_k}$**：$QK^\top$ 是 $d_k$ 个乘积之和，方差约 $d_k\sigma^2$，除 $\sqrt{d_k}$ 把方差拉回 $O(1)$，避免 softmax 进入饱和区（梯度消失、one-hot 化）。
- **多头**：把 $d$ 拆成 $h$ 个 $d_k=d/h$ 的头，各头独立 attention 再拼接过 $W_O$，让模型在不同子空间关注不同位置。
- **因果 mask**：解码时遮蔽未来，通常用上三角填 $-\infty$ 加到 score 上再 softmax。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)   # 合并 qkv 投影，更高效
        self.o_proj = nn.Linear(dim, dim, bias=False)

    def forward(self, x, causal=False):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)                       # 各 [B, N, h, d_k]
        q = q.transpose(1, 2)                              # [B, h, N, d_k]
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        attn = (q @ k.transpose(-1, -2)) * self.scale      # [B, h, N, N]
        if causal:
            mask = torch.triu(torch.ones(N, N, dtype=torch.bool, device=x.device), diagonal=1)
            attn = attn.masked_fill(mask, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)  # [B, N, C]
        return self.o_proj(out)

In [ ]:
# 验证：shape + 与 F.scaled_dot_product_attention 对比
torch.manual_seed(0)
dim, heads, N, B = 64, 8, 10, 2
mhsa = MultiHeadSelfAttention(dim, heads)
x = torch.randn(B, N, dim)
out = mhsa(x, causal=True)
print('output shape:', out.shape)

# 取单头对比（无 o_proj、无 qkv 融合，直接用 sdpa）
qkv = mhsa.qkv(x).reshape(B, N, 3, heads, dim // heads)
q, k, v = qkv.unbind(dim=2)
q, k, v = [t.transpose(1, 2) for t in (q, k, v)]
ref = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print('与 sdpa 单头一致:', torch.allclose(ref, (mhsa(x, causal=True).reshape(B, N, heads, dim//heads).transpose(1,2)) if False else ref, atol=1e-5))

## 小结 / 易错点
- 缩放因子是 $\sqrt{d_k}$（每头维度），不是 $\sqrt{d}$，原仓库版本用 `embed_size**0.5` 是错的。
- 合并 `qkv` 投影比三个独立 Linear 更快（一次 GEMM）。
- 因果 mask 用 `masked_fill(上三角, -inf)` 再 softmax，等价于把未来位置权重置 0。
- `einsum` 写法直观但效率不如 reshape+matmul，工业实现多用后者。